# 06 — Graph Neural Network Preparation

This notebook prepares the generated knowledge graph for possible Graph Neural Network experiments.

The goal is not to train a full GNN yet, but to show how the project knowledge graph can be converted into a machine-learning-ready graph representation.

This notebook includes:

- loading the exported knowledge graph,
- inspecting graph statistics,
- analyzing node and edge types,
- creating node ID mappings,
- creating edge index arrays,
- building simple node features,
- preparing a lightweight graph dataset structure.

This provides a clear foundation for future experiments with GCN, GraphSAGE, GAT, node classification, or link prediction.

## Imports and setup

In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()

for candidate in [current, *current.parents]:
    if (candidate / "src").exists() and (candidate / "config.yaml").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find project root")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from src.utils.common import setup_notebook

CONFIG, PATHS = setup_notebook()

## Load knowledge graph

In [ ]:
graph_path = PATHS.data_graphs / "knowledge_graph.graphml"

if not graph_path.exists():
    raise FileNotFoundError(
        f"Knowledge graph not found at {graph_path}. "
        "Run the pipeline first: python -m src.pipeline.run_pipeline --skip-embeddings"
    )

G = nx.read_graphml(graph_path)

print("Graph loaded:", graph_path)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

## Basic graph statistics

In [ ]:
graph_stats = {
    "num_nodes": G.number_of_nodes(),
    "num_edges": G.number_of_edges(),
    "density": nx.density(G),
    "is_directed": G.is_directed(),
    "num_connected_components": (
        nx.number_weakly_connected_components(G)
        if G.is_directed()
        else nx.number_connected_components(G)
    ),
}

graph_stats_df = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in graph_stats.items()]
)

display(graph_stats_df)

## Node type analysis

In [ ]:
def get_node_type(data):
    return (
        data.get("type")
        or data.get("node_type")
        or data.get("label")
        or "UNKNOWN"
    )

node_rows = []

for node_id, data in G.nodes(data=True):
    node_rows.append(
        {
            "node_id": node_id,
            "node_type": get_node_type(data),
            **data,
        }
    )

nodes_df = pd.DataFrame(node_rows)

display(nodes_df.head())

## Node type distribution

In [ ]:
node_type_counts = (
    nodes_df["node_type"]
    .value_counts()
    .reset_index()
)

node_type_counts.columns = ["node_type", "count"]

display(node_type_counts)

plt.figure(figsize=(8, 5))
plt.bar(node_type_counts["node_type"], node_type_counts["count"])
plt.title("Knowledge Graph Node Type Distribution")
plt.xlabel("Node Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Edge type analysis

In [ ]:
def get_edge_type(data):
    return (
        data.get("relation")
        or data.get("type")
        or data.get("edge_type")
        or "RELATED_TO"
    )

edge_rows = []

for source, target, data in G.edges(data=True):
    edge_rows.append(
        {
            "source": source,
            "target": target,
            "edge_type": get_edge_type(data),
            "weight": float(data.get("weight", 1.0)),
        }
    )

edges_df = pd.DataFrame(edge_rows)

display(edges_df.head())

## Edge type distribution

In [ ]:
edge_type_counts = (
    edges_df["edge_type"]
    .value_counts()
    .reset_index()
)

edge_type_counts.columns = ["edge_type", "count"]

display(edge_type_counts)

plt.figure(figsize=(8, 5))
plt.bar(edge_type_counts["edge_type"], edge_type_counts["count"])
plt.title("Knowledge Graph Edge Type Distribution")
plt.xlabel("Edge Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Centrality analysis

In [ ]:
degree_centrality = nx.degree_centrality(G)

try:
    pagerank = nx.pagerank(G)
except Exception:
    pagerank = {node: 0.0 for node in G.nodes()}

centrality_df = pd.DataFrame(
    [
        {
            "node_id": node,
            "node_type": get_node_type(G.nodes[node]),
            "degree": G.degree(node),
            "degree_centrality": degree_centrality.get(node, 0.0),
            "pagerank": pagerank.get(node, 0.0),
        }
        for node in G.nodes()
    ]
).sort_values("pagerank", ascending=False)

display(centrality_df.head(20))

## Visualize top central nodes

In [ ]:
top_n = min(30, len(centrality_df))

top_nodes = centrality_df.head(top_n)["node_id"].tolist()
subgraph = G.subgraph(top_nodes).copy()

plt.figure(figsize=(14, 10))

pos = nx.spring_layout(
    subgraph,
    seed=CONFIG.get("project", {}).get("seed", 42),
    k=1.2,
)

node_sizes = [
    300 + 80 * subgraph.degree(node)
    for node in subgraph.nodes()
]

labels = {
    node: str(node)[:35] + "..." if len(str(node)) > 35 else str(node)
    for node in subgraph.nodes()
}

nx.draw_networkx_edges(subgraph, pos, alpha=0.3)
nx.draw_networkx_nodes(subgraph, pos, node_size=node_sizes)
nx.draw_networkx_labels(subgraph, pos, labels=labels, font_size=8)

plt.title("Top Central Nodes in the Knowledge Graph")
plt.axis("off")
plt.tight_layout()
plt.show()

## Create node index mapping

In [ ]:
node_to_idx = {
    node_id: index
    for index, node_id in enumerate(G.nodes())
}

idx_to_node = {
    index: node_id
    for node_id, index in node_to_idx.items()
}

print("Number of indexed nodes:", len(node_to_idx))

list(node_to_idx.items())[:10]

##  Create edge index

In [ ]:
edge_index = np.array(
    [
        [node_to_idx[source], node_to_idx[target]]
        for source, target in G.edges()
    ],
    dtype=np.int64,
).T

print("edge_index shape:", edge_index.shape)
print(edge_index[:, :10])

## Create node type encoding

In [ ]:
node_types = sorted(nodes_df["node_type"].unique())

node_type_to_idx = {
    node_type: index
    for index, node_type in enumerate(node_types)
}

node_type_features = np.zeros(
    (len(node_to_idx), len(node_type_to_idx)),
    dtype=np.float32,
)

for node_id, data in G.nodes(data=True):
    node_type = get_node_type(data)
    node_index = node_to_idx[node_id]
    type_index = node_type_to_idx[node_type]
    node_type_features[node_index, type_index] = 1.0

print("Node type feature matrix shape:", node_type_features.shape)
print("Node type mapping:", node_type_to_idx)

## Add structural node features

In [ ]:
degree_values = np.array(
    [G.degree(node_id) for node_id in G.nodes()],
    dtype=np.float32,
).reshape(-1, 1)

pagerank_values = np.array(
    [pagerank.get(node_id, 0.0) for node_id in G.nodes()],
    dtype=np.float32,
).reshape(-1, 1)

# Normalize degree
if degree_values.max() > 0:
    degree_values = degree_values / degree_values.max()

if pagerank_values.max() > 0:
    pagerank_values = pagerank_values / pagerank_values.max()

structural_features = np.concatenate(
    [
        degree_values,
        pagerank_values,
    ],
    axis=1,
)

print("Structural feature matrix shape:", structural_features.shape)

## Final node feature matrix

In [ ]:
node_features = np.concatenate(
    [
        node_type_features,
        structural_features,
    ],
    axis=1,
)

print("Final node feature matrix shape:", node_features.shape)

## Prepare GNN dataset dictionary

In [ ]:
gnn_dataset = {
    "num_nodes": len(node_to_idx),
    "num_edges": edge_index.shape[1],
    "node_to_idx": node_to_idx,
    "idx_to_node": idx_to_node,
    "node_type_to_idx": node_type_to_idx,
    "edge_index": edge_index,
    "node_features": node_features,
}

print("GNN-ready graph dataset prepared.")
print("Nodes:", gnn_dataset["num_nodes"])
print("Edges:", gnn_dataset["num_edges"])
print("Node features:", gnn_dataset["node_features"].shape)
print("Edge index:", gnn_dataset["edge_index"].shape)

## Save GNN-ready tables

In [ ]:
output_dir = PATHS.data_graphs / "gnn"
output_dir.mkdir(parents=True, exist_ok=True)

nodes_export = nodes_df.copy()
nodes_export["node_index"] = nodes_export["node_id"].map(node_to_idx)

edges_export = edges_df.copy()
edges_export["source_index"] = edges_export["source"].map(node_to_idx)
edges_export["target_index"] = edges_export["target"].map(node_to_idx)

nodes_export.to_csv(output_dir / "gnn_nodes.csv", index=False)
edges_export.to_csv(output_dir / "gnn_edges.csv", index=False)

np.save(output_dir / "edge_index.npy", edge_index)
np.save(output_dir / "node_features.npy", node_features)

print("Saved GNN outputs to:", output_dir)

## Optional PyTorch Geometric conversion

In [ ]:
try:
    import torch
    from torch_geometric.data import Data

    pyg_data = Data(
        x=torch.tensor(node_features, dtype=torch.float32),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
    )

    print(pyg_data)

except ImportError:
    print(
        "PyTorch Geometric is not installed. "
        "The graph has still been prepared as NumPy arrays and CSV files."
    )

## Possible GNN Experiments

The prepared graph representation can be used for several future experiments.

### 1. Node Type Classification

Goal:

Predict whether a node is a `PAPER`, `AUTHOR`, `TOPIC`, or `CONCEPT`.

Inputs:

- node type one-hot features,
- degree,
- PageRank,
- graph connectivity.

Limitation:

This is mostly a sanity-check task because node type is already encoded in the features.

### 2. Link Prediction

Goal:

Predict missing relationships between papers, concepts, authors, or topics.

This would be more useful for the project because it could support knowledge graph completion.

Example:

- Does a paper belong to a missing topic?
- Is a concept related to another concept?
- Should two papers be connected based on shared concepts?

### 3. Graph Embedding Learning

Goal:

Learn node embeddings from the graph structure and compare them with Sentence Transformer embeddings.

This could connect the graph-based and text-based parts of the project.

### 4. Retrieval Reranking

Goal:

Use learned graph embeddings as an additional signal for retrieval reranking.

This would extend the current KG-enhanced retrieval approach.

## Interpretation

This notebook shows that the generated knowledge graph can be transformed into a GNN-ready representation.

The graph was converted into:

- indexed nodes,
- edge index matrix,
- node type features,
- structural node features,
- exportable CSV and NumPy files.

This means the project is prepared for future Graph Neural Network experiments.